# Gamma Scalping and Volatility Analysis

## 1. Project Objective

This notebook implements and evaluates a delta-hedged SPX straddle strategy using E-mini S&P 500 (ES) futures as the hedge instrument.

The analysis focuses on how gamma gains, theta decay, implied versus realised (subsequently) volatility, hedge frequency and transaction costs interact to determine strategy P&L (profit and loss).

In [13]:
# ES Gamma Scalping (Daily) — March 10–20, 2020

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statistics import NormalDist

In [14]:
plt.rcParams["figure.figsize"] = (10, 4) # set as default plot size

## 2. Model Configuration

This strategy models a long SPX straddle dynamically delta-hedged using E-mini S&P 500 futures. The configuration below defines the contract specifications, strategy rules, risk controls, option assumptions and backtest period. Contract-specific March and June ES futures are represented separately so that the hedge roll can be handled explicitly without altering the strategy mechanics.

In [ ]:
# Contract specifications
SPX_OPTION_MULTIPLIER = 100.0
ES_POINT_VALUE        = 50.0

# Strategy specification
N_STRADDLES           = 1
DELTA_REHEDGE_BAND    = 0.10
COMMISSION_PER_ES     = 1.25

# Risk controls
DAILY_PROFIT_TARGET   = 2000.0
DAILY_DRAWDOWN_LIMIT  = -2000.0

# Option assumptions
RISK_FREE_RATE        = 0.0
OPTION_EXPIRY         = pd.Timestamp("2020-03-20")
STRIKE_INTERVAL       = 5.0

# Market data
SPX_TICKER            = "^GSPC"
VOL_TICKER            = "^VIX9D"

# Contract-specific ES hedge data
ES_MARCH_CONTRACT     = "ESH20"
ES_JUNE_CONTRACT      = "ESM20"
ES_ROLL_DATE          = pd.Timestamp("2020-03-12")
ES_DATA_FILE          = "data/es_futures_mar2020.csv"

# Backtest period
INTERVAL              = "1d"
START                 = "2020-03-10"
END                   = "2020-03-21"  # yfinance end date is exclusive

## 3. Data collection and preparation

SPX and short-dated implied-volatility observations are aligned with contract-specific March and June 2020 E-mini S&P 500 futures data over the backtest period. SPX provides the underlying level for the option position, while ES provides the hedge instrument.

The straddle strike is fixed at inception and time to expiry declines throughout the option life. March and June ES contracts are retained separately around the roll so that changes between different futures contracts are not incorrectly recorded as hedge P&L.

In [ ]:
def download_close_series(ticker, column_name):
    """Download a clean daily closing-price series."""

    raw = yf.download(
        ticker,
        start=START,
        end=END,
        interval=INTERVAL,
        auto_adjust=False,
        progress=False,
    )

    if raw is None or raw.empty:
        raise RuntimeError(f"No data returned for {ticker}.")

    if isinstance(raw.columns, pd.MultiIndex):
        if "Close" not in raw.columns.get_level_values(0):
            raise RuntimeError(f"No Close column returned for {ticker}.")

        close = raw["Close"]

        if isinstance(close, pd.DataFrame):
            close = close.iloc[:, 0]

    else:
        if "Close" not in raw.columns:
            raise RuntimeError(f"No Close column returned for {ticker}.")

        close = raw["Close"]

    close = pd.to_numeric(
        close,
        errors="coerce"
    ).dropna()

    close.name = column_name
    close.index = pd.to_datetime(close.index)

    if close.index.tz is not None:
        close.index = close.index.tz_localize(None)

    close.index = close.index.normalize()

    return close.sort_index()

In [ ]:
def load_es_contract_data(filepath):
    """Load and validate the contract-specific ES futures dataset."""

    es_data = pd.read_csv(
        filepath,
        parse_dates=["Date"],
    )

    required_columns = [
        "Date",
        "ESH20_Fixing",
        "ESH20_Settlement",
        "ESH20_Volume",
        "ESH20_OpenInterest",
        "ESM20_Fixing",
        "ESM20_Settlement",
        "ESM20_Volume",
        "ESM20_OpenInterest",
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in es_data.columns
    ]

    if missing_columns:
        raise RuntimeError(
            "Missing required ES data columns: "
            + ", ".join(missing_columns)
        )

    es_data = (
        es_data
        .set_index("Date")
        .sort_index()
    )

    es_data.index = (
        pd.to_datetime(es_data.index)
        .tz_localize(None)
        .normalize()
    )

    numeric_columns = [
        column
        for column in required_columns
        if column != "Date"
    ]

    es_data[numeric_columns] = (
        es_data[numeric_columns]
        .apply(pd.to_numeric, errors="coerce")
    )

    return es_data

In [ ]:
# =========================
# MARKET DATA PREPARATION
# =========================

# Market series used by the option model.
spx = download_close_series(
    SPX_TICKER,
    "SPX"
)

vol_proxy = download_close_series(
    VOL_TICKER,
    "VIX9D"
)


# Contract-specific ES hedge data.
es_contracts = load_es_contract_data(
    ES_DATA_FILE
)


# Align all inputs to common trading dates.
market = pd.concat(
    [
        spx,
        vol_proxy,
        es_contracts,
    ],
    axis=1,
    join="inner",
).dropna(
    subset=["SPX", "VIX9D"]
).sort_index()


# Restrict observations to the option's life.
market = market.loc[
    (market.index >= pd.Timestamp(START))
    & (market.index <= OPTION_EXPIRY)
].copy()

if market.empty:
    raise RuntimeError(
        "No common market observations remain "
        "after aligning the input datasets."
    )


# =========================
# OPTION SPECIFICATION
# =========================

# Fix the straddle strike once at inception.
initial_spx = market["SPX"].iloc[0]

STRIKE = (
    round(initial_spx / STRIKE_INTERVAL)
    * STRIKE_INTERVAL
)

market["K"] = STRIKE


# VIX9D is quoted in percentage points.
market["sigma"] = (
    market["VIX9D"] / 100.0
)


# Time to expiry declines using calendar time.
market["days_to_expiry"] = (
    OPTION_EXPIRY.normalize()
    - market.index
).days

market["T"] = (
    market["days_to_expiry"] / 365.0
).clip(lower=0.0)


# =========================
# ES CONTRACT ROLL
# =========================

market["ES_contract"] = np.where(
    market.index < ES_ROLL_DATE,
    ES_MARCH_CONTRACT,
    ES_JUNE_CONTRACT,
)

market["is_roll_date"] = (
    market.index == ES_ROLL_DATE
)


# Active fixing observation.
market["ES_fixing"] = np.where(
    market.index < ES_ROLL_DATE,
    market["ESH20_Fixing"],
    market["ESM20_Fixing"],
)


# Active settlement observation.
market["ES_settlement"] = np.where(
    market.index < ES_ROLL_DATE,
    market["ESH20_Settlement"],
    market["ESM20_Settlement"],
)


# =========================
# SAME-CONTRACT PRICE CHANGES
# =========================

march_fixing_change = (
    market["ESH20_Fixing"].diff()
)

june_fixing_change = (
    market["ESM20_Fixing"].diff()
)

market["ES_fixing_change"] = np.where(
    market.index < ES_ROLL_DATE,
    march_fixing_change,
    june_fixing_change,
)


march_settlement_change = (
    market["ESH20_Settlement"].diff()
)

june_settlement_change = (
    market["ESM20_Settlement"].diff()
)

market["ES_settlement_change"] = np.where(
    market.index < ES_ROLL_DATE,
    march_settlement_change,
    june_settlement_change,
)


# =========================
# VARIABLES FOR LATER ANALYSIS
# =========================

market["SPX_log_return"] = np.log(
    market["SPX"]
    / market["SPX"].shift(1)
)


# =========================
# DIAGNOSTICS
# =========================

print(f"Backtest observations: {len(market)}")
print(f"Initial SPX level:      {initial_spx:.2f}")
print(f"Straddle strike:        {STRIKE:.0f}")
print(f"Option expiry:          {OPTION_EXPIRY.date()}")
print(f"ES roll date:           {ES_ROLL_DATE.date()}")

display(
    market[
        [
            "SPX",
            "VIX9D",
            "sigma",
            "K",
            "days_to_expiry",
            "T",
            "ES_contract",
            "ES_fixing",
            "ES_settlement",
        ]
    ]
)

HTTP Error 404: 

1 Failed download:
['ESH20.CME']: YFTzMissingError('possibly delisted; no timezone found')


RuntimeError: No data returned for ESH20.CME.

## 4. Option pricing and Greeks

Black-Scholes is used to value the call and put forming the long SPX straddle and to calculate the position's delta, gamma and theta at each observation.

Options values and Grreks are calculated in SPX index-point terms in this section. The SPX contract mutliplier is applied later in the backtest when translating these exposures into dollar P&L and determining the required ES futures hedge.

At expiry, the call and put are valued at intrinsic value and the remaining gamma and theta exposures are set to zero. This avoids numerical instability as time to expiry approaches zero and allows the strategy to be evaluated using actual option repricing, while the Greeks are retained for hedge construction and P&L attribution.

In [ ]:
# BLACK–SCHOLES GREEKS
# =========================
N = NormalDist(0, 1)

def d1(S, K, r, sigma, T):
    return (np.log(S/K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))

def d2(d1_, sigma, T):
    return d1_ - sigma * np.sqrt(T)

def gamma_bs(S, K, r, sigma, T):
    d1_ = d1(S, K, r, sigma, T)
    pdf = np.exp(-0.5 * d1_**2) / np.sqrt(2*np.pi)
    return pdf / (S * sigma * np.sqrt(T))

def delta_call(S, K, r, sigma, T): return N.cdf(d1(S, K, r, sigma, T))
def delta_put(S, K, r, sigma, T):  return N.cdf(d1(S, K, r, sigma, T)) - 1

def theta_call(S, K, r, sigma, T):
    d1_ = d1(S, K, r, sigma, T); d2_ = d2(d1_, sigma, T)
    pdf = np.exp(-0.5 * d1_**2) / np.sqrt(2*np.pi)
    return -(S * pdf * sigma) / (2 * np.sqrt(T)) - r*K*np.exp(-r*T)*N.cdf(d2_)

def theta_put(S, K, r, sigma, T):
    d1_ = d1(S, K, r, sigma, T); d2_ = d2(d1_, sigma, T)
    pdf = np.exp(-0.5 * d1_**2) / np.sqrt(2*np.pi)
    return -(S * pdf * sigma) / (2 * np.sqrt(T)) + r*K*np.exp(-r*T)*N.cdf(-d2_)

def straddle_greeks(S, K, sigma, T, n=1.0):
    d_c = delta_call(S, K, RISK_FREE, sigma, T)
    d_p = delta_put (S, K, RISK_FREE, sigma, T)
    g   = gamma_bs  (S, K, RISK_FREE, sigma, T)
    t   = theta_call(S, K, RISK_FREE, sigma, T) + theta_put(S, K, RISK_FREE, sigma, T)
    return n*(d_c+d_p), n*2.0*g, n*t

## 5. Delta-hedged Gamma-scalping backtest

In [ ]:
# BACKTEST LOOP (DAILY SAFE)
# =========================
records = []
q_fut = 0.0
prev_delta, prev_gamma, prev_theta = None, None, None
prev_S, curr_day, daily_pnl = None, None, 0.0

for row in es.itertuples(index=True):
    ts, S, sigma, K, T, day = row.Index, row.ES, row.sigma, row.K, row.T, row.day
    if prev_S is None:
        curr_day = day
        d, g, t = straddle_greeks(S, K, sigma, T, N_STRADDLES)
        prev_delta, prev_gamma, prev_theta = d, g, t
        prev_S = S
        continue

    dS = S - prev_S
    delta_step = prev_delta * dS
    gamma_step = 0.5 * prev_gamma * (dS**2)
    theta_step = prev_theta * dt_year
    hedge_step = q_fut * ES_POINT_VALUE * dS
    pnl_step   = delta_step + gamma_step + theta_step + hedge_step
    daily_pnl += pnl_step

    event = "HOLD"; commission = 0.0
    if (daily_pnl >= DAILY_PROFIT_TARGET) or (daily_pnl <= DAILY_DRAWDOWN_LIMIT):
        q_fut = 0.0; event = "STOP"
    else:
        d_new, g_new, t_new = straddle_greeks(S, K, sigma, T, N_STRADDLES)
        net_delta = d_new + q_fut * ES_POINT_VALUE
        denom = max(1e-6, abs(d_new))
        if (abs(net_delta)/denom) > DELTA_REHEDGE_BAND:
            q_target   = -d_new / ES_POINT_VALUE
            dq         = q_target - q_fut
            commission = COMMISSION_PER_ES * abs(dq)
            q_fut      = q_target
            event      = "REHEDGE"
        prev_delta, prev_gamma, prev_theta = d_new, g_new, t_new

    records.append([ts, S, delta_step, gamma_step, theta_step,
                    hedge_step, -commission, pnl_step, daily_pnl, q_fut, event])

    prev_S = S


## 6. Results

In [ ]:
# RESULTS
# =========================
res_cols = ["ts","S","delta_step","gamma_step","theta_step",
            "hedge_pnl","commission","pnl_step","daily_pnl","q_fut","event"]
res = pd.DataFrame(records, columns=res_cols)
res["ts"] = pd.to_datetime(res["ts"])
res = res.set_index("ts").sort_index()
res["gamma_scalp"] = res["delta_step"] + res["gamma_step"]

daily = res.groupby(pd.Grouper(freq="1D")).agg(
    gamma_scalp = ("gamma_scalp","sum"),
    theta       = ("theta_step","sum"),
    hedge_pnl   = ("hedge_pnl","sum"),
    commissions = ("commission","sum"),
    pnl_total   = ("daily_pnl","last"),
    rehedges    = ("event", lambda s: (s == "REHEDGE").sum())
)
daily["cum_pnl"] = daily["pnl_total"].cumsum()

print("PnL units: USD")
print(daily)
print("Total PnL USD:", round(daily["pnl_total"].sum(), 2))
print("Total Rehedges:", int(daily["rehedges"].sum()))

## 7. Visualisation

In [ ]:
# PLOTS
# =========================
plt.figure()
plt.bar(daily.index, daily["pnl_total"])
plt.title("Daily PnL (USD)")
plt.xlabel("Date"); plt.ylabel("PnL (USD)")
plt.tight_layout(); plt.show()

plt.figure()
plt.plot(daily.index, daily["cum_pnl"], marker="o")
plt.title("Cumulative PnL (USD)")
plt.xlabel("Date"); plt.ylabel("Cumulative PnL (USD)")
plt.tight_layout(); plt.show()

ax = daily[["gamma_scalp","theta","hedge_pnl","commissions"]].plot(kind="bar", figsize=(12,5), stacked=True)
ax.set_title("Daily PnL Components (USD)")
ax.set_xlabel("Date"); ax.set_ylabel("USD")
plt.tight_layout(); plt.show()


## 8. Limitations, Extensions